In [ ]:
from __future__ import annotations
import re, csv, hashlib, unicodedata, json
from pathlib import Path
from collections import defaultdict
import pandas as pd
from tqdm import tqdm
from charset_normalizer import from_path

DATA_DIR   = Path("~/Desktop/NLP_Submission/Data/1_Scraped_data/scraped_twitch")
OUTPUT_DIR = Path("~/Desktop/NLP_Submission/Data/2_Merged_data/twitch_merged")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "twitch_merged.csv"

COMMON_ENCODINGS = [
    "utf-8", "utf-8-sig", "utf-16", "latin1", "cp1252",
    "shift_jis", "gb18030", "big5",
]

IRC_RE = re.compile(
    r"^:(?P<author>[^!]+)!.*?PRIVMSG\s+#\S+\s+:(?P<msg>.*)$",
    re.DOTALL,
)

TIMESTAMP_COL = "timestamp"                       
BASE_COLS     = ["author", "message"]             
KEY_COLS      = BASE_COLS + [TIMESTAMP_COL]       

In [ ]:
def md5(path: Path, block: int = 1 << 20) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

def detect_encoding(path: Path) -> str | None:
    try:
        best = from_path(str(path)).best()
    except Exception:
        return None
    if not best or not best.encoding:
        return None

    if hasattr(best, "confidence"):                
        conf = float(best.confidence)
    else:                                           
        fp = best.fingerprint
        if isinstance(fp, dict):
            conf = float(fp.get("confidence", 0))
        elif isinstance(fp, str):
            try:
                conf = float(json.loads(fp).get("confidence", 0))
            except Exception:
                conf = 0.0
        else:
            conf = 0.0
    return best.encoding if conf >= 0.20 else None


# Tries several readers, skipping malformed rows (`on_bad_lines="skip"`):
# 1. C-engine with COMMON_ENCODINGS
# 2. Python engine + QUOTE_NONE with the same encodings
# 3. Auto-detected encoding from charset-normalizer
# 4. Raw line fallback (every line → 'message' col)
# Returns DataFrame or None.
def read_csv_resilient(path: Path, **kwargs) -> pd.DataFrame | None:
    for enc in COMMON_ENCODINGS:                    
        try:
            return pd.read_csv(
                path, engine="c", encoding=enc,
                on_bad_lines="skip", low_memory=False, **kwargs
            )
        except Exception:
            pass
    for enc in COMMON_ENCODINGS:                   
        try:
            return pd.read_csv(
                path, engine="python", quoting=csv.QUOTE_NONE,
                encoding=enc, sep=",",
                on_bad_lines="skip", low_memory=False, **kwargs
            )
        except Exception:
            pass
    auto_enc = detect_encoding(path)                
    if auto_enc:
        try:
            return pd.read_csv(
                path, engine="python", quoting=csv.QUOTE_NONE,
                encoding=auto_enc, sep=",",
                on_bad_lines="skip", low_memory=False, **kwargs
            )
        except Exception:
            pass
    try:                                       
        enc = auto_enc or "utf-8"
        with path.open("r", encoding=enc, errors="replace") as fh:
            lines = [ln.rstrip("\n") for ln in fh]
        return pd.DataFrame({"message": lines})
    except Exception:
        pass
    print(f"  Skipped unreadable file: {path.name}")
    return None


# Extracts 'author' and cleaned 'message' from raw IRC lines
# Ensures a 'timestamp' column exists (NaN if missing)
# Drops duplicated headers if present

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    if "message" not in df.columns:
        raise KeyError("Column 'message' not found.")

    df = df.loc[:, ~df.columns.duplicated(keep="first")]

    ext = df["message"].str.extract(IRC_RE, expand=True)
    df["author"]  = ext["author"]
    df["message"] = ext["msg"].str.strip()

    if TIMESTAMP_COL not in df.columns:
        df[TIMESTAMP_COL] = pd.NA

    df = df.dropna(subset=["author", "message"]).reset_index(drop=True)
    keep_other = [c for c in df.columns if c not in BASE_COLS + [TIMESTAMP_COL]]
    return df[KEY_COLS + keep_other]


def canonical_key(row: pd.Series) -> str:
    norm = lambda s: unicodedata.normalize("NFC", str(s)).strip().lower()
    ts = row.get(TIMESTAMP_COL, pd.NA)
    if isinstance(ts, pd.Series):   
        ts = ts.iloc[0]
    ts_val = str(ts) if pd.notna(ts) else f"missing-{row.name}"
    return f"{norm(row['author'])}|{norm(row['message'])}|{ts_val}"

In [ ]:
files = sorted(DATA_DIR.glob("*.csv"))
print(f"  {len(files)} CSV files found in {DATA_DIR}")

size_groups = defaultdict(list)
for p in files:
    size_groups[p.stat().st_size].append(p)

unique_files: list[Path] = []
print("\n▸ Removing byte-identical copies …")
for same_size in tqdm(size_groups.values(), unit="file"):
    seen_hashes: set[str] = set()
    for p in same_size:
        h = md5(p)
        if h not in seen_hashes:
            seen_hashes.add(h)
            unique_files.append(p)
print(f"  {len(unique_files)} unique files remain after step 1")

In [ ]:
print("\n▸ Merging files and de-duplicating rows …")
merged_frames = []
seen_rows: set[str] = set()

for p in tqdm(sorted(unique_files), unit="file"):
    df_raw = read_csv_resilient(p)
    if df_raw is None or df_raw.empty:
        continue
    try:
        df_clean = clean_dataframe(df_raw)
    except Exception as e:
        print(f"  Skipped {p.name}: {e}")
        continue

    df_clean["__key__"] = df_clean.apply(canonical_key, axis=1)
    new = df_clean[~df_clean["__key__"].isin(seen_rows)]
    seen_rows.update(new["__key__"])
    merged_frames.append(new.drop(columns="__key__"))

In [ ]:
full = pd.concat(merged_frames, ignore_index=True)
full.to_csv(OUTPUT_FILE, index=False)

print(
    f"\n  Saved {OUTPUT_FILE}\n"
    f"   • Rows written : {len(full):,}\n"
    f"   • Files merged : {len(unique_files)}"
)